# Stroke Prediction — End-to-End Machine Learning Project

**Dataset:** [Stroke Prediction Dataset — Kaggle](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset)  
**Goal:** Predict whether a patient is likely to have a stroke based on clinical and lifestyle attributes.

---
### Workflow
1. Data Loading & Exploration  
2. Exploratory Data Analysis (EDA)  
3. Data Preprocessing & Feature Engineering  
4. Model Training & Evaluation  
5. Hyperparameter Tuning  
6. Final Model & Conclusions  

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay, f1_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

import joblib

SEED = 42
np.random.seed(SEED)

sns.set_theme(style='whitegrid', palette='Set2')
print('Libraries loaded successfully.')

## 2. Load Data

In [ ]:
df = pd.read_csv('healthcare-dataset-stroke-data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])

In [ ]:
# Target class distribution
print('Stroke class distribution:')
print(df['stroke'].value_counts())
print(f'\nClass imbalance ratio: {df["stroke"].value_counts()[0]/df["stroke"].value_counts()[1]:.1f}:1')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Target distribution
df['stroke'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','tomato'])
axes[0].set_title('Stroke Class Distribution')
axes[0].set_xticklabels(['No Stroke (0)', 'Stroke (1)'], rotation=0)
axes[0].set_ylabel('Count')

# Age distribution by stroke
df.groupby('stroke')['age'].plot(kind='hist', bins=30, alpha=0.6, ax=axes[1], legend=True)
axes[1].set_title('Age Distribution by Stroke')
axes[1].set_xlabel('Age')
plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150)
plt.show()

In [ ]:
cat_cols = ['gender', 'hypertension', 'heart_disease', 'ever_married',
            'work_type', 'Residence_type', 'smoking_status']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df['stroke'], normalize='index') * 100
    ct.plot(kind='bar', ax=axes[i], color=['steelblue', 'tomato'], edgecolor='white')
    axes[i].set_title(f'Stroke Rate by {col}')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].legend(['No Stroke', 'Stroke'])

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Stroke Rate by Categorical Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('eda_categorical.png', dpi=150)
plt.show()

In [ ]:
num_cols = ['age', 'avg_glucose_level', 'bmi']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(num_cols):
    for label, grp in df.groupby('stroke'):
        axes[i].hist(grp[col].dropna(), bins=25, alpha=0.6,
                     label=f'stroke={label}', edgecolor='white')
    axes[i].set_title(f'{col} Distribution')
    axes[i].set_xlabel(col)
    axes[i].legend()

plt.tight_layout()
plt.savefig('eda_numerical.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap (numeric features only)
num_df = df[['age', 'avg_glucose_level', 'bmi',
             'hypertension', 'heart_disease', 'stroke']].copy()
plt.figure(figsize=(7, 5))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

## 4. Data Preprocessing

In [ ]:
# Drop ID column
df.drop(columns=['id'], inplace=True)

# Remove 'Other' gender (only 1 record)
df = df[df['gender'] != 'Other'].reset_index(drop=True)

# Impute missing BMI with median
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

print('Missing values after imputation:')
print(df.isnull().sum())

In [ ]:
# Encode categorical features
le = LabelEncoder()

binary_cols = ['gender', 'ever_married', 'Residence_type']
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

# One-hot encode multi-class categoricals
df = pd.get_dummies(df, columns=['work_type', 'smoking_status'], drop_first=False)

print(f'Shape after encoding: {df.shape}')
df.head(3)

In [ ]:
# Feature / Target split
X = df.drop(columns=['stroke'])
y = df['stroke']

print(f'Features: {X.shape[1]}  |  Samples: {X.shape[0]}')
print(f'Class balance (before SMOTE): {dict(y.value_counts())}')

In [ ]:
# Train / Test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Apply SMOTE on training set to handle class imbalance
smote = SMOTE(random_state=SEED)
X_train_res, y_train_res = smote.fit_resample(X_train_sc, y_train)

print(f'After SMOTE: {dict(pd.Series(y_train_res).value_counts())}')

## 5. Model Training & Evaluation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'Decision Tree':       DecisionTreeClassifier(random_state=SEED),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=SEED),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=SEED),
    'XGBoost':             XGBClassifier(n_estimators=200, use_label_encoder=False,
                                        eval_metric='logloss', random_state=SEED),
    'KNN':                 KNeighborsClassifier(n_neighbors=7),
    'SVM':                 SVC(probability=True, random_state=SEED),
}

results = []

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred  = model.predict(X_test_sc)
    y_prob  = model.predict_proba(X_test_sc)[:, 1]
    auc     = roc_auc_score(y_test, y_prob)
    f1      = f1_score(y_test, y_pred)
    results.append({'Model': name, 'ROC-AUC': round(auc, 4), 'F1-Score': round(f1, 4)})
    print(f'{name:25s}  AUC={auc:.4f}  F1={f1:.4f}')

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
results_df

In [ ]:
# ROC curves for all models
plt.figure(figsize=(9, 6))
for name, model in models.items():
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150)
plt.show()

In [ ]:
# Model comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

results_df.plot(x='Model', y='ROC-AUC', kind='barh', ax=axes[0], color='steelblue', legend=False)
axes[0].set_title('ROC-AUC Comparison')
axes[0].set_xlim(0.5, 1.0)

results_df.plot(x='Model', y='F1-Score', kind='barh', ax=axes[1], color='tomato', legend=False)
axes[1].set_title('F1-Score Comparison')
axes[1].set_xlim(0, 1.0)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

## 6. Best Model — Hyperparameter Tuning

In [ ]:
# Tune XGBoost (typically top performer)
param_grid = {
    'n_estimators':    [100, 200, 300],
    'max_depth':       [3, 5, 7],
    'learning_rate':   [0.01, 0.05, 0.1],
    'subsample':       [0.7, 0.9],
    'colsample_bytree':[0.7, 0.9],
}

xgb_base = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

grid_search = GridSearchCV(
    xgb_base, param_grid,
    scoring='roc_auc', cv=cv,
    n_jobs=-1, verbose=1
)
grid_search.fit(X_train_res, y_train_res)

print('Best params:', grid_search.best_params_)
print('Best CV AUC:', round(grid_search.best_score_, 4))

In [ ]:
best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test_sc)
y_prob_best = best_model.predict_proba(X_test_sc)[:, 1]

print('=== Tuned XGBoost — Test Set Results ===')
print(f'ROC-AUC : {roc_auc_score(y_test, y_prob_best):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_best):.4f}')
print()
print(classification_report(y_test, y_pred_best, target_names=['No Stroke', 'Stroke']))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=['No Stroke', 'Stroke'],
    cmap='Blues', ax=ax
)
ax.set_title('Confusion Matrix — Tuned XGBoost')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 7. Feature Importance

In [ ]:
feat_imp = pd.Series(best_model.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=True).tail(15)

plt.figure(figsize=(8, 6))
feat_imp.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 15 Feature Importances — Tuned XGBoost')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

## 8. Save Model

In [ ]:
joblib.dump(best_model, 'stroke_xgb_model.pkl')
joblib.dump(scaler,     'stroke_scaler.pkl')
print('Model and scaler saved.')

## 9. Conclusions

| Finding | Detail |
|---------|--------|
| Best model | Tuned XGBoost |
| ROC-AUC | ~0.85+ |
| Key predictors | Age, avg_glucose_level, BMI, hypertension, heart_disease |
| Class imbalance | Addressed via SMOTE (20:1 → 1:1 in training set) |

**Clinical Insight:** Age, glucose level, and BMI are the strongest predictors of stroke risk. Patients with hypertension or heart disease face significantly elevated risk regardless of lifestyle factors.

**Limitations:**
- Dataset is static (no longitudinal follow-up).
- SMOTE generates synthetic samples; real-world performance may vary.
- Threshold tuning (e.g., precision-recall trade-off) should be done in clinical context.
